[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Exceptions as Classes


## What you will be able to do

Design a family of your own exceptions under one base class, decide what each one carries beyond its
message, catch exactly the failures you mean to handle and no others, and turn a low-level error into
one that says what went wrong in your program's terms.


## The idea

### The problem

A loader reads station readings from a file, and two things go wrong in practice. A sensor glitch
sometimes records `999`. That happens, so the reading should be skipped and counted, and the run
should carry on. Now and then a file arrives corrupted, with a line such as `warm` where a number
should be. That file cannot be trusted, and the run should stop.

The obvious way to report the glitch is `raise ValueError(...)`, and the obvious way to skip it is
`except ValueError:`. The trouble is that `float("warm")` raises `ValueError` too. The handler
written for glitches catches the corruption as well, counts it as one more skipped reading, and the
run finishes with a file it should have refused. Nothing in the output says so.

The other workaround is to read the message, and skip the error only if its text contains a
particular word. That works until somebody rewords the message.

Built-in exceptions describe what went wrong in Python's terms: a value was the wrong kind. A program
needs to say what went wrong in its own terms: a reading was impossible, or a file was corrupt. The
**Errors and Exceptions** notebook showed that `class SensorError(Exception):` is enough to make a new
kind of error. This notebook designs several, so that each can be caught on its own.

### What an exception class is

> An **exception class** is an ordinary class that inherits from `Exception`, directly or through
> another exception class. `raise` signals a failure with an object of it, and `except` catches by
> class: the class it names, and every subclass of that class. A **family** of exception classes under
> one base lets a caller choose how precisely to catch, either one specific failure or anything the
> code can raise.

### Why it works that way

`except ReadingError` is an `isinstance` check, the one from the **Inheritance** notebook, so the
shape of the family decides what every `except` can say. The base class is the handle for "anything
from this code", and each subclass is one specific failure. Give the glitch its own class, and
`except ReadingError` catches glitches and nothing else, while `float`'s `ValueError` goes past
untouched.

An exception is an object, so it can carry data. A `ReadingError` can store the value
and the line it came from as attributes, and a handler can act on those instead of picking them out
of the message. The message itself is built once, in `__init__`, and handed to `super().__init__`,
which is what `str(error)` prints.

A new exception can also have two parents. `ReadingError(StationError, ValueError)` is caught by code
written for the new family and by older code that already catches `ValueError`, which is how the
standard library's own `json.JSONDecodeError` fits in.

And when a low-level error causes a higher-level one, `raise ... from error` keeps the original
attached, so the caller sees the problem in the program's terms without losing Python's.

### Where you will meet this

`json.JSONDecodeError` carries the line and column of the mistake, and `FileNotFoundError` carries the
filename. Libraries of any size define families of their own, and their documentation names the base
class to catch when you want everything they can raise.

### What this notebook covers

The loader raising `ValueError` against the loader raising `ReadingError`, through the same numbered
steps. Then the hierarchy you have been catching all along, a family of your own, what to put in an
exception, a second parent, why checking the message is fragile, clause order, and turning one error
into another. Then a loader that uses all of it.

### A first look

An exception that carries its own facts. There is nothing to run yet: read it, and read the output
underneath it.

```python
class ReadingError(ValueError):
    def __init__(self, value, line):
        super().__init__(f"line {line}: {value} is outside -90 to 60")
        self.value = value
        self.line = line


try:
    raise ReadingError(999.0, line=4)
except ReadingError as error:
    print(error)
    print("the value was", error.value, "on line", error.line)
```

```
line 4: 999.0 is outside -90 to 60
the value was 999.0 on line 4
```

The handler read the value and the line as attributes, without taking the message apart.


## Setup

One import.

- `json` supplies one of the standard library's own exception classes, `JSONDecodeError`, used in
  one section

Every exception class in this notebook is written in the section that uses it.

**Run this cell before the rest of the notebook.**


In [1]:
import json

print("ready")


ready


## Worked examples

### Before and after: raising `ValueError`, or a class of your own

Here is the problem from the top of this notebook, in code. Each version has a `check` function that
refuses an impossible value, and a `load` function that skips whatever `check` refuses, returning the
readings it kept and a count of those it skipped. Both versions go through the same three steps:

1. Load three good readings. All three should be kept, none skipped.
2. Load a sensor glitch, `999`. It should be skipped and counted, and the run should go on.
3. Load a corrupt line, `"warm"`. The run should stop with an error.

First, `check` raises `ValueError`, so `load` catches `ValueError`.


In [2]:
def check(celsius):
    if not -90 <= celsius <= 60:
        raise ValueError(f"{celsius} is outside -90 to 60")
    return celsius


def load(lines):
    kept, skipped = [], 0
    for line in lines:
        try:
            kept.append(check(float(line)))
        except ValueError:
            skipped += 1
    return kept, skipped


The three steps.


In [3]:
# 1. Load three good readings. All three should be kept, none skipped.
print("1.", load(["-4.1", "-2.6", "-3.8"]))

# 2. Load a sensor glitch, 999. It should be skipped and counted, and the run should go on.
print("2.", load(["-4.1", "999", "-2.6"]))

# 3. Load a corrupt line, "warm". The run should stop with an error.
try:
    print("3.", load(["-4.1", "warm", "-2.6"]))
except ValueError as error:
    print("3. stopped:", error)


1. ([-4.1, -2.6, -3.8], 0)
2. ([-4.1, -2.6], 1)
3. ([-4.1, -2.6], 1)


Step 3 should have stopped, and it did not. `float("warm")` raised a `ValueError` of its own, the
handler written for glitches caught it, and the corrupt line was counted as one more skipped reading.

Look at the results of steps 2 and 3 together: they are identical. After this `load`, a glitch and a
corrupt file cannot be told apart, by the program or by anyone reading its output.

Now the glitch gets a class of its own. `ReadingError` still inherits from `ValueError`, because an
impossible reading is a kind of bad value. The change that matters is in `load`, which now catches
`ReadingError` and nothing else.


In [4]:
class ReadingError(ValueError):
    """A reading that could not have been measured on Earth."""


def check(celsius):
    if not -90 <= celsius <= 60:
        raise ReadingError(f"{celsius} is outside -90 to 60")
    return celsius


def load(lines):
    kept, skipped = [], 0
    for line in lines:
        try:
            kept.append(check(float(line)))
        except ReadingError:
            skipped += 1
    return kept, skipped


The same three steps, with the code unchanged.


In [5]:
# 1. Load three good readings. All three should be kept, none skipped.
print("1.", load(["-4.1", "-2.6", "-3.8"]))

# 2. Load a sensor glitch, 999. It should be skipped and counted, and the run should go on.
print("2.", load(["-4.1", "999", "-2.6"]))

# 3. Load a corrupt line, "warm". The run should stop with an error.
try:
    print("3.", load(["-4.1", "warm", "-2.6"]))
except ValueError as error:
    print("3. stopped:", error)


1. ([-4.1, -2.6, -3.8], 0)
2. ([-4.1, -2.6], 1)
3. stopped: could not convert string to float: 'warm'


Step 3 stopped, as it should. `load` caught only `ReadingError`, so `float`'s `ValueError` went past
it and out of `load`, and the calling code reported Python's own message about the corrupt line.
Steps 1 and 2 behaved exactly as before.

| | `check` raises `ValueError` | `check` raises `ReadingError` |
|---|---|---|
| What `load` catches | `ValueError`, from `check` and from `float` | `ReadingError`, from `check` only |
| Step 2, a glitch | skipped and counted | skipped and counted |
| Step 3, a corrupt line | skipped and counted, like a glitch | stopped the run |
| Telling steps 2 and 3 apart | impossible: the results are identical | the corrupt line raises |

The rest of this notebook takes the idea apart.

| Question | The section that answers it |
|---|---|
| Which exceptions have I been catching all along? | The hierarchy you have been catching |
| How do I design errors that can be caught separately or together? | A family of your own |
| What should an exception carry besides its message? | What to put in an exception |
| How does a new exception stay catchable as a `ValueError`? | Two parents |
| Why not just check the message text? | Why not check the message? |
| How does a corrupt line become a clear error of my own? | Turning one error into another |

### The hierarchy you have been catching

Every built-in exception sits in a hierarchy, and `__mro__` shows each one's line of ancestors.


In [6]:
for cls in (ValueError, KeyError, IndexError, FileNotFoundError, json.JSONDecodeError):
    print(f"{cls.__name__:<18}", [c.__name__ for c in cls.__mro__])

print()
for attempt in (lambda: {}["missing"], lambda: [][3]):
    try:
        attempt()
    except LookupError as error:
        print("except LookupError caught", type(error).__name__)


ValueError         ['ValueError', 'Exception', 'BaseException', 'object']
KeyError           ['KeyError', 'LookupError', 'Exception', 'BaseException', 'object']
IndexError         ['IndexError', 'LookupError', 'Exception', 'BaseException', 'object']
FileNotFoundError  ['FileNotFoundError', 'OSError', 'Exception', 'BaseException', 'object']
JSONDecodeError    ['JSONDecodeError', 'ValueError', 'Exception', 'BaseException', 'object']

except LookupError caught KeyError
except LookupError caught IndexError


`KeyError` and `IndexError` share a parent, `LookupError`, so one `except LookupError` catches both.
And `JSONDecodeError` is a `ValueError`: the standard library gave JSON its own error while keeping it
catchable by code that already handled bad values.

The standard library's exceptions also carry facts beyond their message.


In [7]:
try:
    json.loads('{"name": "Tromso",}')
except json.JSONDecodeError as error:
    print("msg:   ", error.msg)
    print("lineno:", error.lineno, "| colno:", error.colno)

try:
    open("no/such/readings.csv")
except FileNotFoundError as error:
    print("filename:", error.filename, "| strerror:", error.strerror)


msg:    Illegal trailing comma before end of object
lineno: 1 | colno: 18
filename: no/such/readings.csv | strerror: No such file or directory


A program that reports a JSON mistake can point at line 1, column 18, without reading the message, and
one that reports a missing file can name the file. That is the model for the exceptions you write.

### A family of your own

One base class for everything the station code raises, and one subclass for each failure a handler
might want to treat differently.


In [8]:
class StationError(Exception):
    """Anything that goes wrong in the station code."""


class ReadingError(StationError):
    """A reading that could not have been measured."""


class UnitError(StationError):
    """A unit the station does not support."""


def trigger(kind):
    if kind == "reading":
        raise ReadingError("999 is outside -90 to 60")
    raise UnitError("'K' is not a supported unit")


for kind in ("reading", "unit"):
    try:
        trigger(kind)
    except StationError as error:
        print(f"except StationError caught {type(error).__name__}: {error}")

try:
    trigger("unit")
except ReadingError:
    print("not reached")
except UnitError as error:
    print("except ReadingError let it pass, and except UnitError caught it:", error)


except StationError caught ReadingError: 999 is outside -90 to 60
except StationError caught UnitError: 'K' is not a supported unit
except ReadingError let it pass, and except UnitError caught it: 'K' is not a supported unit


`except StationError` caught both, because both are kinds of `StationError`. `except ReadingError`
caught only its own class and let the `UnitError` go past to the next clause. A caller can handle one
failure precisely, or everything from the station code at once, and the family decides which classes
each choice covers.

### What to put in an exception

The message is for people. A handler that needs to act, by logging the line or correcting the value,
needs the facts themselves, and an exception can carry them as attributes.


In [9]:
class ReadingError(StationError):
    """A reading that could not have been measured."""

    def __init__(self, value, line):
        super().__init__(f"line {line}: {value} is outside -90 to 60")
        self.value = value
        self.line = line


try:
    raise ReadingError(999.0, line=4)
except ReadingError as error:
    print("str(error): ", error)
    print("error.value:", error.value)
    print("error.line: ", error.line)
    print("error.args: ", error.args)


str(error):  line 4: 999.0 is outside -90 to 60
error.value: 999.0
error.line:  4
error.args:  ('line 4: 999.0 is outside -90 to 60',)


`__init__` builds the message once and passes it to `super().__init__`, which stores it in `args`,
where `str(error)` finds it. The facts go on the object as ordinary attributes. The quiet error at the
end of this notebook shows what happens when the `super().__init__` line is left out.

### Two parents

A new exception often replaces a built-in one that callers already catch. Giving it both parents keeps
those callers working.


In [10]:
class ReadingError(StationError, ValueError):
    """A reading that could not have been measured."""

    def __init__(self, value, line):
        super().__init__(f"line {line}: {value} is outside -90 to 60")
        self.value = value
        self.line = line


print([c.__name__ for c in ReadingError.__mro__])
print()

for handler in (StationError, ValueError):
    try:
        raise ReadingError(999.0, line=4)
    except handler as error:
        print(f"except {handler.__name__} caught it: {error}")


['ReadingError', 'StationError', 'ValueError', 'Exception', 'BaseException', 'object']

except StationError caught it: line 4: 999.0 is outside -90 to 60
except ValueError caught it: line 4: 999.0 is outside -90 to 60


The same exception is caught by the new family's handler and by an older `except ValueError`. That is
the choice `JSONDecodeError` made, and it is the usual one for a new error that narrows an existing
kind.

### Why not check the message?

The workaround from the top of this notebook is to catch `ValueError` and look at the text. Here it is,
with the wording of the message passed in, so that the effect of rewording it can be seen.


In [11]:
def check_worded(celsius, wording):
    if not -90 <= celsius <= 60:
        raise ValueError(f"{celsius} {wording}")
    return celsius


def load_by_message(lines, wording):
    kept, skipped = [], 0
    for line in lines:
        try:
            kept.append(check_worded(float(line), wording))
        except ValueError as error:
            if "outside" in str(error):
                skipped += 1
            else:
                raise
    return kept, skipped


print("original wording:", load_by_message(["-4.1", "999"], "is outside -90 to 60"))

try:
    load_by_message(["-4.1", "999"], "is not a possible temperature")
except ValueError as error:
    print("reworded:        stopped on", error)


original wording: ([-4.1], 1)
reworded:        stopped on 999.0 is not a possible temperature


With the original wording the glitch was skipped. After somebody improved the message, the word
`outside` was gone, the check no longer matched, and a glitch stopped the run. Nothing about the code
that checks the text changed; only a message somewhere else did. A class is a name the code can rely on
in a way message text is not.

### Clause order, with a family of your own

The **Errors and Exceptions** notebook put it plainly: a general type placed first catches everything,
and the specific clauses after it never run. With a family you designed, the base class is the general
type.


In [12]:
def handle(error):
    try:
        raise error
    except StationError:
        return "stopped the run"
    except ReadingError:
        return "skipped the reading"


def handle_fixed(error):
    try:
        raise error
    except ReadingError:
        return "skipped the reading"
    except StationError:
        return "stopped the run"


print("base class first:", handle(ReadingError(999.0, line=4)))
print("subclass first:  ", handle_fixed(ReadingError(999.0, line=4)))


base class first: stopped the run
subclass first:   skipped the reading


With the base class first, a glitch stopped the run, because a `ReadingError` is a `StationError` and
the first clause matched. No error was raised to point this out. List the most specific class first,
and the base class last, as a catch for everything else in the family.

### Turning one error into another

`float("warm")` raises Python's `ValueError`, which says what went wrong in Python's terms. A caller of
the loader wants to know which line of the file was bad. Catch the low-level error and raise one of
your own `from` it.


In [13]:
class CorruptLineError(StationError):
    """A line that is not a number at all."""

    def __init__(self, text, line):
        super().__init__(f"line {line}: {text!r} is not a number")
        self.text = text
        self.line = line


def parse(text, line):
    try:
        return float(text)
    except ValueError as error:
        raise CorruptLineError(text, line) from error


try:
    parse("warm", 3)
except CorruptLineError as error:
    print("raised:   ", error)
    print("__cause__:", type(error.__cause__).__name__, "-", error.__cause__)


raised:    line 3: 'warm' is not a number
__cause__: ValueError - could not convert string to float: 'warm'


The caller gets an error that names the line, in the loader's terms. `from error` attached the original
as `__cause__`, so nothing was lost: an uncaught error of this kind prints both, with the original first.
Writing `from None` instead detaches it, for the rare case where the original would only confuse.

### Putting it together: a loader that knows what went wrong

A family with one base, two failures that need different handling, attributes that carry the facts, a
second parent where it helps, and a low-level error turned into one of the family's own. `load` skips
glitches, keeping the errors so it can report each one, and lets a corrupt line stop the run.


In [14]:
class StationError(Exception):
    """Base class for everything the station code raises."""


class ReadingError(StationError, ValueError):
    """A value that could not have been measured. Skip it, and count it."""

    def __init__(self, value, line):
        super().__init__(f"line {line}: {value} is outside -90 to 60")
        self.value = value
        self.line = line


class CorruptLineError(StationError):
    """A line that is not a number at all. The file cannot be trusted."""

    def __init__(self, text, line):
        super().__init__(f"line {line}: {text!r} is not a number")
        self.text = text
        self.line = line


def parse(text, line):
    try:
        celsius = float(text)
    except ValueError as error:
        raise CorruptLineError(text, line) from error
    if not -90 <= celsius <= 60:
        raise ReadingError(celsius, line)
    return celsius


def load(lines):
    kept, skipped = [], []
    for line, text in enumerate(lines, start=1):
        try:
            kept.append(parse(text, line))
        except ReadingError as error:
            skipped.append(error)
    return kept, skipped


kept, skipped = load(["-4.1", "999", "-2.6", "-120", "-3.8"])

print("kept:", kept)
for error in skipped:
    print(f"skipped line {error.line}: {error.value}")


kept: [-4.1, -2.6, -3.8]
skipped line 2: 999.0
skipped line 4: -120.0


Two glitches were skipped, and because each `ReadingError` carries its line and value, the report names
both without reading a single message. Now a corrupt file.


In [15]:
try:
    load(["-4.1", "warm", "-2.6"])
except StationError as error:
    print(f"stopped: {type(error).__name__}: {error}")
    print(f"because: {type(error.__cause__).__name__}: {error.__cause__}")


stopped: CorruptLineError: line 2: 'warm' is not a number
because: ValueError: could not convert string to float: 'warm'


The run stopped on line 2, with an error in the loader's terms and Python's original attached. The
caller caught `StationError`, the base, which covers every failure the station code can raise, and
`load` itself caught only `ReadingError`, which is why the corruption got out.

### Where each part came from

| In the loader | What it relies on | The section that showed it |
|---|---|---|
| glitches skipped and corruption stopping the run | catch the specific class, not the built-in one | Before and after |
| `StationError` above both failures | one base class for a family | A family of your own |
| `error.line` and `error.value` in the report | an exception can carry data as attributes | What to put in an exception |
| `ReadingError` also a `ValueError` | two parents keep older handlers working | Two parents |
| no message text read anywhere | a class is reliable where wording is not | Why not check the message? |
| `CorruptLineError` with `__cause__` | `raise ... from error` | Turning one error into another |
| the caller catching `StationError`, the base | one handler for anything the family raises | A family of your own |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/13-exceptions-as-classes-solutions.ipynb).

**1.** Define `InstrumentError` and two subclasses of it, `CalibrationError` and `DisconnectedError`.
Raise each one inside a `try`, and catch both with a single `except InstrumentError`.


In [16]:
# your code here


**2.** Give `CalibrationError` an `__init__(self, instrument, offset)` that builds a message with
`super().__init__` and stores both attributes. Raise one, and print the message and each attribute.


In [17]:
# your code here


**3.** Make `DisconnectedError` a subclass of `ConnectionError` as well as `InstrumentError`, and show
that an `except ConnectionError` catches it.


In [18]:
# your code here


**4.** Write `read_value(text)`, which turns the `ValueError` from `float(text)` into a
`BadValueError` of your own with `raise ... from error`. Print the new error and its `__cause__`.


In [19]:
# your code here


**5.** Write two `except` clauses in the wrong order, the base class first, and show which one runs for
a `CalibrationError`. Then put them in the right order and show it again.


In [20]:
# your code here


**6.** Write `total(readings)`, which adds up readings, skipping any value over 100 by catching the
`CalibrationError` that a `check` function raises for it, while letting a `DisconnectedError`, raised
for a reading of `None`, stop the whole call.


In [21]:
# your code here


## Common errors

### TypeError: raising something that is not an exception

`raise` only accepts objects whose class inherits from `BaseException`, which every exception class does
through `Exception`.


In [22]:
class NotAnException:
    """Meant to be an error, but inherits from nothing."""


raise NotAnException()


TypeError: exceptions must derive from BaseException

`exceptions must derive from BaseException` means the class is missing its parent. Write
`class NotAnException(Exception):`, and it can be raised and caught like any other.

### TypeError: raising an exception without the arguments its `__init__` needs

Once an exception class has its own `__init__`, raising it has to supply what that `__init__` takes.


In [23]:
raise ReadingError


TypeError: ReadingError.__init__() missing 2 required positional arguments: 'value' and 'line'

`raise ReadingError` with no parentheses creates the object with no arguments, and this `__init__` needs
a value and a line. The message names both. Raise it as `ReadingError(999.0, line=4)`.

### An error escapes an `except` that seems to name it

To catch two classes in one clause, they go in a tuple. Joining them with `or` looks similar and does
something else.


In [24]:
class UnitError(StationError):
    """A unit the station does not support."""


try:
    raise UnitError("'K' is not a supported unit")
except ReadingError or UnitError:
    print("caught")


UnitError: 'K' is not a supported unit

`ReadingError or UnitError` is an ordinary expression, and `or` returns the first of the two that
counts as true, which is `ReadingError`, since a class always does. So the clause meant
`except ReadingError`, and the `UnitError` went straight past it. The tuple form catches either.


In [25]:
try:
    raise UnitError("'K' is not a supported unit")
except (ReadingError, UnitError) as error:
    print("caught", type(error).__name__, "-", error)


caught UnitError - 'K' is not a supported unit


### The quiet one: an `__init__` that never passes the message on

An exception class whose `__init__` builds a message and forgets to hand it to `super().__init__`
raises and catches perfectly well. It just loses the message.


In [26]:
class Forgot(StationError):
    def __init__(self, value, line):
        message = f"line {line}: {value} is outside -90 to 60"
        self.value = value
        self.line = line


try:
    raise Forgot(999.0, 4)
except Forgot as error:
    print("skipped:", error)
    print("args:   ", error.args)


skipped: (999.0, 4)
args:    (999.0, 4)


No error, and the log line says `skipped: (999.0, 4)`. The message was built and never used. Python
stores whatever arguments the class was called with in `args`, and with nothing else to go on,
`str(error)` prints them.

Anybody reading that log learns nothing about what went wrong. Pass the message on, and it comes back.


In [27]:
class Remembered(StationError):
    def __init__(self, value, line):
        super().__init__(f"line {line}: {value} is outside -90 to 60")
        self.value = value
        self.line = line


try:
    raise Remembered(999.0, 4)
except Remembered as error:
    print("skipped:", error)


skipped: line 4: 999.0 is outside -90 to 60


## Recap

- An exception class inherits from `Exception`, directly or through another exception class.
- `except` catches by class: the class it names and every subclass of that class.
- A family under one base lets callers catch one specific failure, or anything the code raises.
- A failure raised as a plain `ValueError` is caught along with every other `ValueError`, including
  Python's own.
- Give each failure that a handler needs to tell apart a class of its own.
- An exception can carry data as attributes, so a handler need not take the message apart.
- Build the message in `__init__` and pass it to `super().__init__`, or `str(error)` shows the raw
  arguments instead.
- Two parents keep a new exception catchable by older handlers, as `JSONDecodeError` is a `ValueError`.
- Checking the message text breaks as soon as the wording changes.
- List the most specific `except` first. The first clause that matches wins.
- `except (A, B)` catches either class. `except A or B` catches only `A`.
- `raise NewError(...) from error` keeps the original as `__cause__`.


## What is next

The **Worked Designs** notebook, which puts this guide to work. Shapes are written twice, first with
plain functions and then with classes, side by side, so that the case for each is made by running code
rather than by argument. A playlist and a set of notification channels then show the rest of the guide
working together, and you design a real-world object of your own. It returns to the question the
**Why Classes** notebook opened with, of when a class is the right tool at all.


---

&#8592; **Previous:** [Interfaces](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/12-interfaces.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Worked Designs](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/14-worked-designs.ipynb) &#8594;
